In [2]:
from getpass import getpass

HUGGINGFACEHUB_API_TOKEN = getpass()

In [3]:
import os
os.environ["HUGGINGFACEHUB_API_TOKEN"] = HUGGINGFACEHUB_API_TOKEN
os.environ["CONNECTION_STRING"] = "sqlite:///../database_test.db"

In [5]:
import os
from typing import Optional
from sqlalchemy import create_engine, text
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

SYSTEM_PROMPT = """
You are a database engine who contains the following tables and their schemas. 
Your task is to generate the SQL statements necessary that resemble the user's query. You must generate only and only SQL for SQLite. Don't add any preamble or explanation along the SQL statement. You must return just a single line with a single query.

Tables:

Schema for table article:
  id (INTEGER) // Primary key, the numerical id of the article within the database
  path (VARCHAR) // The path of the archive file that contains the files associated to the article
  pmcid (VARCHAR) // PMC identifier, a unique identifier assigned to each article in PMC OA
  pmid (VARCHAR) // PubMed identifier, a unique identifier assigned to each article in PubMed. This identifier is different from the PMC identifier
  last_updated (DATETIME) // Time stamp of the last time the article was updated
  journal_id (INTEGER) // Foreign key to the journal table. Use this key to join to the journal table when you need to retrieve the journal's name
  year (INTEGER)	// Year of the article's publication
  month (VARCHAR) // Month or month range of the article's publication
  day (INTEGER) // Day of the month of the article's publication
  volume (INTEGER) // Volume of the journal in which the article was published
  issue (INTEGER) // Issue of the journal in which the article was published
  eaccession (VARCHAR) // Electronic accession number of the article, This can be the page number, the range of the page numbers, a DOI, or any other identifier
  license_id (INTEGER) // Foreign key to the license table. Use this key to join to the license table when you need to retrieve the license's name
  retracted (BOOLEAN) // Indicates whether the article has been retracted or not

Schema for table journal:
  id (INTEGER) // Primary key, the numerical id of the journal within the database
  commercial (BOOLEAN) // Indicates whether the journal is a commercial journal or not
  name (VARCHAR) // The name of the journal. This is the name that is used to refer to the journal in the article's citation

Schema for table license:
  id (INTEGER) // Primary key, the numerical id of the license within the database
  name (VARCHAR) // The name of the license under which the article is distributed

""".strip()

# Generate SQL from a user query
prompt_template = ChatPromptTemplate.from_messages(
	[("system", SYSTEM_PROMPT),
	("user", """User's query in natural language: {query}
				SQL statement:""".strip())]
)

# Read your HF API Key from the environment
HUGGINGFACEHUB_API_TOKEN = os.environ["HUGGINGFACEHUB_API_TOKEN"]

# Create and endpoint client for Qwen 2.5 Coder 32B hosted in HF's serverless inference API
endpoint = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-Coder-32B-Instruct",
    huggingfacehub_api_token=HUGGINGFACEHUB_API_TOKEN,
)

# Wrap the endpoint with a "Chat" object, suitable for langchain pipelines
llm = ChatHuggingFace(llm=endpoint)

# Create a pipeline that takes a user query, generates SQL, and parses the SQL into a string
chain = prompt_template | llm | StrOutputParser()

In [6]:
chain.invoke({"query": "The distinct names of journals in the database that have had at least three thousand publications in 2020"})

'SELECT DISTINCT j.name FROM journal j JOIN article a ON j.id = a.journal_id WHERE a.year = 2020 GROUP BY j.name HAVING COUNT(a.id) >= 3000'

In [7]:
def execute_query(query:str, connection_string: Optional[str] = None) -> None:
	""" Generates a SQL statement from a user query and executes it """
	if not connection_string:
		connection_string = os.getenv("CONNECTION_STRING")
	engine = create_engine(connection_string)

	ret = []
	with engine.connect() as conn:
		result = conn.execute(text(query))
		for row in result:
			ret.append(row)

	return ret

In [8]:
from langchain_core.runnables import RunnableLambda

executable_chain = chain | RunnableLambda(execute_query)

In [9]:
executable_chain.invoke({"query": "The distinct names of journals in the database that have had at least three thousand publications in 2020"})

[('ACS Omega',),
 ('BMJ Open',),
 ('Cancers (Basel)',),
 ('Front Immunol',),
 ('Front Microbiol',),
 ('Front Psychol',),
 ('Innov Aging',),
 ('Int J Environ Res Public Health',),
 ('Int J Mol Sci',),
 ('J Clin Med',),
 ('Materials (Basel)',),
 ('Medicine (Baltimore)',),
 ('Molecules',),
 ('Nat Commun',),
 ('Nutrients',),
 ('PLoS One',),
 ('Polymers (Basel)',),
 ('Sci Rep',),
 ('Sensors (Basel)',)]

In [10]:
execute_query(chain.invoke({"query": "Give me a histogram of the amount of papers published each year since 2020"}))

[(2020, 555632),
 (2021, 693211),
 (2022, 789091),
 (2023, 709180),
 (2024, 669032),
 (2025, 21282)]

In [11]:
execute_query(chain.invoke({"query": "How many articles are indexed in the data base?"}))

[(6592687,)]